## Notebook para runtime de testes do modelo

Este notebook serve apenas como "servidor" temporário para rodar o modelo em desenvolvimento. O limite de GPU do colab é de até 12h/ notebook / conta, então deve ser gerado um novo notebook ao término do limite.

### Carregamento do modelo

In [1]:
# instalação de dependências para runtime
!pip install -U transformers accelerate huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 25.2 MB/s eta 0:00:00


In [14]:
# logando no hugging face
from google.colab import userdata

from huggingface_hub import login, HfApi
login(token=f"{userdata.get('HF_TOKEN')}")  # logar com access token do hugging face para a org


In [16]:
# carregando modelo
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "fiap-hospital-helper/hospital-helper-qwen2.5-1.5b"
MODEL_REVISION = "v2.0" #tag referente à última versão do modelo treinado

api = HfApi()
info = api.model_info(MODEL_ID, revision="v2.0")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, force_download=True, torch_dtype=torch.bfloat16, device_map="auto")

print("SHA da tag v2.0 :", info.sha)
print("Commit carregado:", model.config._commit_hash)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

SHA da tag v2.0 : a4f2670102c2ed5762dfc9a58953568c6d1113fe
Commit carregado: a4f2670102c2ed5762dfc9a58953568c6d1113fe


In [17]:
# teste rápido para validação do carregamento

prompt = (
    "### Instrucao:\nResponda em pt-BR usando o contexto clinico fornecido.\n\n"
    "### Entrada:\nPergunta: Qual o protocolo para dor toracica?\n\n### Resposta:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(resposta)

### Instrucao:
Responda em pt-BR usando o contexto clinico fornecido.

### Entrada:
Pergunta: Qual o protocolo para dor toracica?

### Resposta:
O tratamento depende da causa subjacente. Por exemplo, se a dor é causada por um problema cardíaco ou pulmão, você pode precisar de tratamento médico imediato. Se a dor for temporária e não relacionada a uma doença crônica, você pode ser capaz de lidá-lo sozinho com medicamentos e mudanças no estilo de vida.


### Montagem da API

In [7]:
# instalação de dependências

!pip install fastapi uvicorn pyngrok nest-asyncio -q

In [8]:
# importando token da conta ngrok (criar conta no ngrok para diponibilizar url - muda a cada reinicialização do notebook)
from google.colab import userdata # guardar o token na seção secrets do colab para não ficar hardcoded
ngrok_token = userdata.get('NGROK_TOKEN')

In [9]:
# montando a api
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio, uvicorn
from pyngrok import ngrok

app = FastAPI()

class GenerateRequest(BaseModel):
    pergunta: str
    contexto: str = ""
    max_new_tokens: int = 300

@app.post("/generate")
def generate(req: GenerateRequest):
    prompt = (
        "### Instrucao:\nResponda em pt-BR usando o contexto clinico fornecido.\n\n"
        f"### Entrada:\nPergunta: {req.pergunta}\n"
    )
    if req.contexto:
        prompt += f"Contexto:\n{req.contexto}\n"
    prompt += "\n### Resposta:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=req.max_new_tokens, do_sample=False)
    texto = tokenizer.decode(outputs[0], skip_special_tokens=True)
    resposta = texto.split("### Resposta:")[-1].strip()
    return {"resposta": resposta}

@app.get("/")
def health():
    return {"status": "ok"}

In [ ]:
# criando tunel http para acesso à api
ngrok.set_auth_token(ngrok_token)  # token do ngrok (pegar a url gerada a cada reinicialização do server)

public_url = ngrok.connect(8000)
print("API pública disponível em:", public_url)

nest_asyncio.apply()

config = uvicorn.Config(app, port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

API pública disponível em: NgrokTunnel: "https://shining-setting-trench.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [2852]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     179.118.191.16:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     170.114.6.87:0 - "GET / HTTP/1.1" 200 OK
INFO:     186.208.186.3:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     186.208.186.3:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     186.208.186.3:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     186.208.186.3:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     186.208.186.3:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     186.208.186.3:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     186.208.186.3:0